# T03 — Softmax, escala y saturación

## 1. Título y paper

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017)  
**Fuente primaria:** [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
**Foco de esta miniatura:** por qué √d_k no es cosmética  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)


## 2. Objetivos

1. Ver cómo la escala de los scores cambia la entropía de la atención.
2. Relacionar saturación del softmax con gradientes que se apagan.


## 3. Prerrequisitos

- Python 3.11+ con el paquete instalado (`pip install -e .`).
- Notebook [`P08_transformer`](P08_transformer.ipynb) al menos hojeado.
- Álgebra de vectores: producto escalar, norma y softmax.


## 4. Intuición

El softmax es un mando de contraste. Scores muy grandes → la atención se vuelve un foco que ilumina un solo punto y deja el resto a oscuras (gradiente ≈ 0 para los demás).


## 5. Concepto mínimo

```text
softmax(z)_i = exp(z_i) / Σ_j exp(z_j)
```

Si `q` y `k` tienen componentes independientes de media 0 y varianza 1, `q·k` tiene varianza `d_k`: su magnitud crece como `√d_k`. Dividir por `√d_k` devuelve la varianza a 1.


## 6. Código explicado

Código mínimo, sin dependencias externas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
import math

def softmax(zs):
    m = max(zs)
    e = [math.exp(z - m) for z in zs]
    return [v / sum(e) for v in e]

def entropia(ps):
    return -sum(p * math.log(p + 1e-12) for p in ps)

base = [2.0, 1.0, 0.5, 0.0]
for factor in (0.25, 1, 4, 16):
    p = softmax([z * factor for z in base])
    print(f'escala ×{factor:<3} → {[round(v, 4) for v in p]} · H={entropia(p):.4f}')

## 7. Predicción antes de ejecutar

¿La entropía sube o baja al multiplicar los scores por 16? ¿Qué le pasa al gradiente de los tokens ignorados?

> Escribe tu respuesta antes de continuar.


## 8. Experimento controlado


In [ ]:
import random
rng = random.Random(0)
for d_k in (4, 64, 512):
    q = [rng.gauss(0, 1) for _ in range(d_k)]
    ks = [[rng.gauss(0, 1) for _ in range(d_k)] for _ in range(4)]
    crudos = [sum(a * b for a, b in zip(q, k)) for k in ks]
    escalados = [c / math.sqrt(d_k) for c in crudos]
    print(f'd_k={d_k:>3} · H(sin escala)={entropia(softmax(crudos)):.4f}'
          f' · H(con escala)={entropia(softmax(escalados)):.4f}')

## 9. Salida interpretable

Al crecer `d_k`, la entropía sin escalar se desploma: un token acapara la masa. Con la escala la entropía se mantiene en un rango sano. **La saturación del softmax es el problema; √d_k es la solución.**


## 10. Comentario pedagógico

Esta miniatura aísla **una** pieza del bloque. Aislar es didáctico y también es una simplificación: en el modelo real todas las piezas interactúan y se entrenan juntas.


## 11. Error o anti-patrón deliberado


In [ ]:
grande = [800.0, 799.0]
try:
    print([math.exp(z) for z in grande])
except OverflowError as exc:
    print('OverflowError:', exc, '← softmax ingenuo desborda')

## 12. Corrección


In [ ]:
print('softmax estable (restando el máximo):', [round(p, 6) for p in softmax(grande)])
print('mismo resultado matemático, sin desbordar')

## 13. Desafío guiado

Con d_k=512 y scores sin escalar, calcula el peso del segundo token. ¿Cuánto gradiente le llega?


## 14. Desafío autónomo

Reescribe esta pieza con proyecciones aprendidas y comprueba que tu implementación reproduce las propiedades verificadas aquí (sumas, formas, invariantes). Documenta la semilla.


## 15. Evidencia de aprendizaje

Guarda la salida del experimento, tu predicción previa y una frase sobre qué invariante acabas de verificar.


## 16. Cierre

Pieza cubierta: **por qué √d_k no es cosmética**. Ya puede describirse con precisión, sin metáforas.


## 17. Conexión con el siguiente hito

Con la atención bajo control, se puede aplicar a una secuencia consigo misma: self-attention (T04).
